#Getting Started with PySpark in Google Colab

PySpark is Python interface for Apache Spark. The primary use cases for PySpark are to work with huge amounts of data and for creating data pipelines.

You don't need to work with big data to benefit from PySpark. I find that the SparkSQL is a great tool for performing routine data anlysis. Pandas can get slow and you may find yourself writing a lot of code for data cleaning whereas the same actions take much less code in SQL. Let's get started!

See more here! http://spark.apache.org/docs/latest/api/python/

# 1. Installing PySpark in Google Colab

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("TitanicSparkDemo").getOrCreate()

spark

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,480 kB]
Get:14 https://r2u.sta

In [2]:
spark

In [3]:
import seaborn as sns
import pandas as pd

# Load and filter columns
df = sns.load_dataset("titanic")
df = df[['survived', 'pclass', 'age', 'sex', 'fare']].dropna()

df

,survived,pclass,age,sex,fare
0,0,3,22.0,male,7.2500
1,1,1,38.0,female,71.2833
2,1,3,26.0,female,7.9250
3,1,1,35.0,female,53.1000
4,0,3,35.0,male,8.0500
...,...,...,...,...,...
885,0,3,39.0,female,29.1250
886,0,2,27.0,male,13.0000
887,1,1,19.0,female,30.0000
889,1,1,26.0,male,30.0000


In [4]:
df_spark = spark.createDataFrame(df)


df_spark.printSchema()
df_spark.show()

root
 |-- survived: long (nullable = true)
 |-- pclass: long (nullable = true)
 |-- age: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- fare: double (nullable = true)

+--------+------+----+------+-------+
|survived|pclass| age|   sex|   fare|
+--------+------+----+------+-------+
|       0|     3|22.0|  male|   7.25|
|       1|     1|38.0|female|71.2833|
|       1|     3|26.0|female|  7.925|
|       1|     1|35.0|female|   53.1|
|       0|     3|35.0|  male|   8.05|
|       0|     1|54.0|  male|51.8625|
|       0|     3| 2.0|  male| 21.075|
|       1|     3|27.0|female|11.1333|
|       1|     2|14.0|female|30.0708|
|       1|     3| 4.0|female|   16.7|
|       1|     1|58.0|female|  26.55|
|       0|     3|20.0|  male|   8.05|
|       0|     3|39.0|  male| 31.275|
|       0|     3|14.0|female| 7.8542|
|       1|     2|55.0|female|   16.0|
|       0|     3| 2.0|  male| 29.125|
|       0|     3|31.0|female|   18.0|
|       0|     2|35.0|  male|   26.0|
|       1|     2

In [11]:
# Average age of passengers
df_spark.groupBy("pclass").avg("age").show()

# Count survivors by class
df_spark.groupBy("pclass", "survived").count().show()

+------+------------------+
|pclass|          avg(age)|
+------+------------------+
|     1|38.233440860215055|
|     3| 25.14061971830986|
|     2| 29.87763005780347|
+------+------------------+

+------+--------+-----+
|pclass|survived|count|
+------+--------+-----+
|     3|       0|  270|
|     1|       0|   64|
|     1|       1|  122|
|     2|       0|   90|
|     2|       1|   83|
|     3|       1|   85|
+------+--------+-----+



In [12]:
# make sure you understand what "Views" are in database context

df_spark.createOrReplaceTempView("titanic")


# Query 1: Average age by passenger class
avg_age_query = spark.sql("""
    SELECT Pclass, ROUND(AVG(Age), 2) AS Average_Age
    FROM titanic
    GROUP BY Pclass
    ORDER BY Pclass
""")
print("Average Age by Passenger Class:")
avg_age_query.show()


# Query 2: Count of survivors by passenger class
survival_count_query = spark.sql("""
    SELECT Pclass, Survived, COUNT(*) AS Count
    FROM titanic
    GROUP BY Pclass, Survived
    ORDER BY Pclass, Survived
""")
print("Survival Count by Class:")
survival_count_query.show()

# # Query 3: Average Fare by class and gender
avg_fare_query = spark.sql("""
    SELECT Pclass, Sex, ROUND(AVG(Fare), 2) AS Average_Fare
    FROM titanic
    GROUP BY Pclass, Sex
    ORDER BY Pclass, Sex
""")
print("Average Fare by Class and Gender:")
avg_fare_query.show(truncate=False)



Average Age by Passenger Class:
+------+-----------+
|Pclass|Average_Age|
+------+-----------+
|     1|      38.23|
|     2|      29.88|
|     3|      25.14|
+------+-----------+

Survival Count by Class:
+------+--------+-----+
|Pclass|Survived|Count|
+------+--------+-----+
|     1|       0|   64|
|     1|       1|  122|
|     2|       0|   90|
|     2|       1|   83|
|     3|       0|  270|
|     3|       1|   85|
+------+--------+-----+

Average Fare by Class and Gender:
+------+------+------------+
|Pclass|Sex   |Average_Fare|
+------+------+------------+
|1     |female|107.95      |
|1     |male  |71.14       |
|2     |female|21.95       |
|2     |male  |21.11       |
|3     |female|15.88       |
|3     |male  |12.16       |
+------+------+------------+

